# 07 — Recommendation Testing and Validation

This notebook validates the complete SmartPlay personalized learning-path
recommendation pipeline.

The testing includes:

1. Dataset completeness
2. Student coverage
3. ML prediction consistency
4. Curriculum lesson validity
5. Domain-aware curriculum alignment
6. Learning-path completeness
7. Recommendation status consistency
8. Performance-level consistency
9. Duplicate recommendation detection
10. Student-wise learning-path validation
11. Final recommendation quality report

In [125]:
# ============================================================
# STEP 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("=" * 70)
print("RECOMMENDATION TESTING AND VALIDATION")
print("=" * 70)
print("Libraries loaded successfully.")

RECOMMENDATION TESTING AND VALIDATION
Libraries loaded successfully.


In [126]:
# ============================================================
# STEP 2 — LOAD DATA
# ============================================================

DATA_PATH = Path("../data/processed")

files = {
    "prediction_results": "cv_prediction_results.csv",
    "activity_clean": "activity_clean.csv",
    "assessment_clean": "assessment_clean.csv",
    "curriculum_clean": "curriculum_clean.csv",
    "final_learning_path": "final_personalized_learning_paths.csv",
    "student_summary": "student_learning_path_summary.csv"
}

print("=" * 70)
print("LOADING DATASETS")
print("=" * 70)

loaded_data = {}

for name, filename in files.items():

    file_path = DATA_PATH / filename

    if file_path.exists():

        loaded_data[name] = pd.read_csv(file_path)

        print(f"✓ {filename}")
        print(f"  Shape: {loaded_data[name].shape}")

    else:

        print(f"✗ {filename} NOT FOUND")


print("\nData loading completed.")


# Assign datasets

prediction_results = loaded_data.get(
    "prediction_results"
)

activity_clean = loaded_data.get(
    "activity_clean"
)

assessment_clean = loaded_data.get(
    "assessment_clean"
)

curriculum_clean = loaded_data.get(
    "curriculum_clean"
)

final_learning_path = loaded_data.get(
    "final_learning_path"
)

student_summary = loaded_data.get(
    "student_summary"
)

LOADING DATASETS
✓ cv_prediction_results.csv
  Shape: (15, 5)
✓ activity_clean.csv
  Shape: (60, 12)
✓ assessment_clean.csv
  Shape: (30, 15)
✓ curriculum_clean.csv
  Shape: (20, 9)
✓ final_personalized_learning_paths.csv
  Shape: (30, 22)
✓ student_learning_path_summary.csv
  Shape: (15, 6)

Data loading completed.


In [127]:
# ============================================================
# STEP 3 — DATASET STRUCTURE CHECK
# ============================================================

print("=" * 70)
print("DATASET STRUCTURE CHECK")
print("=" * 70)

datasets = {
    "Prediction Results": prediction_results,
    "Activity Data": activity_clean,
    "Assessment Data": assessment_clean,
    "Curriculum Data": curriculum_clean,
    "Final Learning Path": final_learning_path,
    "Student Summary": student_summary
}

for name, df in datasets.items():

    print("\n" + "-" * 70)
    print(name)

    if df is None:

        print("Dataset not available.")

        continue

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    if "Student_ID" in df.columns:

        print(
            "Unique students:",
            df["Student_ID"].nunique()
        )

    print("Columns:")
    print(list(df.columns))

DATASET STRUCTURE CHECK

----------------------------------------------------------------------
Prediction Results
Rows: 15
Columns: 5
Unique students: 15
Columns:
['Student_ID', 'Actual_Outcome', 'Logistic Regression_Prediction', 'Decision Tree_Prediction', 'Random Forest_Prediction']

----------------------------------------------------------------------
Activity Data
Rows: 60
Columns: 12
Unique students: 15
Columns:
['Activity_ID', 'Student_ID', 'Lesson_ID', 'Activity_Type', 'Difficulty', 'Date', 'Completion_Status', 'Attempts', 'Time_Taken_Min', 'Score', 'Errors', 'Teacher_Evaluation']

----------------------------------------------------------------------
Assessment Data
Rows: 30
Columns: 15
Unique students: 15
Columns:
['Assessment_ID', 'Student_ID', 'Subject', 'Assessment_Type', 'Pre_Test', 'Post_Test', 'Unit_Test', 'Monthly_Test', 'Term_Result', 'Competency_Score', 'Motor_Skills_Score', 'Creativity_Score', 'Interactive_Engagement', 'Learning_Progress', 'Recommended_Action']

--

In [128]:
# ============================================================
# STEP 4 — CREATE TEST FRAMEWORK
# ============================================================

test_results = []


def add_test(
    test_id,
    test_name,
    passed,
    details
):

    test_results.append({

        "Test_ID": test_id,

        "Test_Name": test_name,

        "Status":
            "PASS"
            if passed
            else
            "FAIL",

        "Details": details

    })


def normalize_text(value):

    if pd.isna(value):

        return ""

    return (
        str(value)
        .strip()
        .lower()
    )


print("=" * 70)
print("TEST FRAMEWORK INITIALIZED")
print("=" * 70)

print("✓ Test framework ready.")

TEST FRAMEWORK INITIALIZED
✓ Test framework ready.


In [129]:
# ============================================================
# STEP 5 — TEST STUDENT COVERAGE
# ============================================================

print("=" * 70)
print("TEST 1 — STUDENT COVERAGE")
print("=" * 70)


required_prediction_columns = [
    "Student_ID"
]

required_path_columns = [
    "Student_ID"
]


if (
    prediction_results is not None
    and final_learning_path is not None
    and all(
        col in prediction_results.columns
        for col in required_prediction_columns
    )
    and all(
        col in final_learning_path.columns
        for col in required_path_columns
    )
):

    prediction_students = set(
        prediction_results[
            "Student_ID"
        ]
        .dropna()
        .astype(str)
        .str.strip()
    )

    path_students = set(
        final_learning_path[
            "Student_ID"
        ]
        .dropna()
        .astype(str)
        .str.strip()
    )


    missing_students = (
        prediction_students
        - path_students
    )

    extra_students = (
        path_students
        - prediction_students
    )


    passed = (
        len(missing_students) == 0
        and
        len(extra_students) == 0
    )


    print(
        "Students in prediction results:",
        len(prediction_students)
    )

    print(
        "Students in learning path:",
        len(path_students)
    )

    print(
        "Missing students:",
        sorted(missing_students)
    )

    print(
        "Extra students:",
        sorted(extra_students)
    )


    add_test(
        "T01",
        "Student Coverage",
        passed,
        (
            f"{len(path_students)} students "
            "covered by the learning path"
        )
    )


else:

    add_test(
        "T01",
        "Student Coverage",
        False,
        "Required datasets or columns unavailable"
    )

TEST 1 — STUDENT COVERAGE
Students in prediction results: 15
Students in learning path: 15
Missing students: []
Extra students: []


In [130]:
# ============================================================
# STEP 6 — TEST LEARNING-PATH COUNT
# ============================================================

print("=" * 70)
print("TEST 2 — LEARNING-PATH COUNT")
print("=" * 70)


if final_learning_path is not None:

    student_counts = (
        final_learning_path
        .groupby("Student_ID")
        .size()
    )


    expected_per_student = 2


    incorrect_students = (
        student_counts[
            student_counts
            != expected_per_student
        ]
    )


    total_records = len(
        final_learning_path
    )


    expected_total = (
        len(student_counts)
        *
        expected_per_student
    )


    passed = (
        len(incorrect_students) == 0
        and
        total_records == expected_total
    )


    print(
        "Expected recommendations per student:",
        expected_per_student
    )

    print(
        "Total learning-path records:",
        total_records
    )

    print(
        "Expected total records:",
        expected_total
    )

    print(
        "\nRecommendations per student:"
    )

    print(student_counts)


    print(
        "\nStudents with incorrect "
        "recommendation count:"
    )

    print(incorrect_students)


    add_test(
        "T02",
        "Learning Path Count",
        passed,
        (
            f"{total_records} records; "
            f"{len(incorrect_students)} students "
            "with incorrect counts"
        )
    )


else:

    add_test(
        "T02",
        "Learning Path Count",
        False,
        "Final learning path unavailable"
    )

TEST 2 — LEARNING-PATH COUNT
Expected recommendations per student: 2
Total learning-path records: 30
Expected total records: 30

Recommendations per student:
Student_ID
ST001    2
ST002    2
ST003    2
ST004    2
ST005    2
ST006    2
ST007    2
ST008    2
ST009    2
ST010    2
ST011    2
ST012    2
ST013    2
ST014    2
ST015    2
dtype: int64

Students with incorrect recommendation count:
Series([], dtype: int64)


In [131]:
# ============================================================
# STEP 7 — REQUIRED COLUMN CHECK
# ============================================================

print("=" * 70)
print("TEST 3 — REQUIRED DATASET COLUMNS")
print("=" * 70)


required_columns = [
    "Student_ID",
    "Path_Order",
    "Assessment_ID",
    "Assessment_Subject",
    "Assessment_Type",
    "Recommended_Lesson_ID",
    "Recommended_Subject",
    "Recommended_Topic",
    "Recommended_Competency",
    "Learning_Objective",
    "Difficulty_Level",
    "Required_Skills",
    "Classroom_Activities",
    "Alignment_Score",
    "Alignment_Level",
    "Recommendation_Status",
    "Performance_Level",
    "Overall_Performance",
    "Teaching_Strategy",
    "Recommended_Duration",
    "Teacher_Action"
]


if final_learning_path is not None:

    missing_columns = [
        col
        for col in required_columns
        if col not in final_learning_path.columns
    ]


    passed = (
        len(missing_columns) == 0
    )


    print(
        "Required columns:"
    )

    print(
        required_columns
    )


    print(
        "\nMissing columns:"
    )

    print(
        missing_columns
    )


    if passed:

        print(
            "\n✓ All required learning-path columns are available."
        )

    else:

        print(
            "\n✗ Some required columns are missing."
        )


    add_test(
        "T03",
        "Required Dataset Columns",
        passed,
        (
            "All required columns available"
            if passed
            else
            f"Missing: {missing_columns}"
        )
    )


else:

    add_test(
        "T03",
        "Required Dataset Columns",
        False,
        "Final learning path unavailable"
    )

TEST 3 — REQUIRED DATASET COLUMNS
Required columns:
['Student_ID', 'Path_Order', 'Assessment_ID', 'Assessment_Subject', 'Assessment_Type', 'Recommended_Lesson_ID', 'Recommended_Subject', 'Recommended_Topic', 'Recommended_Competency', 'Learning_Objective', 'Difficulty_Level', 'Required_Skills', 'Classroom_Activities', 'Alignment_Score', 'Alignment_Level', 'Recommendation_Status', 'Performance_Level', 'Overall_Performance', 'Teaching_Strategy', 'Recommended_Duration', 'Teacher_Action']

Missing columns:
[]

✓ All required learning-path columns are available.


In [132]:
# ============================================================
# STEP 8 — TEST CURRICULUM LESSON VALIDITY
# ============================================================

print("=" * 70)
print("TEST 4 — CURRICULUM LESSON VALIDITY")
print("=" * 70)


if (
    final_learning_path is not None
    and curriculum_clean is not None
    and "Recommended_Lesson_ID"
    in final_learning_path.columns
    and "Lesson_ID"
    in curriculum_clean.columns
):

    # --------------------------------------------------------
    # 1. Prepare curriculum lesson IDs
    # --------------------------------------------------------

    curriculum_lessons = (
        curriculum_clean["Lesson_ID"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    curriculum_lesson_set = set(
        curriculum_lessons
    )


    # --------------------------------------------------------
    # 2. Check missing recommendation lesson IDs
    # --------------------------------------------------------

    missing_recommended_id = (
        final_learning_path[
            "Recommended_Lesson_ID"
        ]
        .isna()
    )


    blank_recommended_id = (
        final_learning_path[
            "Recommended_Lesson_ID"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )


    invalid_missing_ids = (
        missing_recommended_id
        |
        blank_recommended_id
    )


    # --------------------------------------------------------
    # 3. Check whether IDs exist
    # --------------------------------------------------------

    recommended_ids = (
        final_learning_path[
            "Recommended_Lesson_ID"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    lesson_exists = (
        recommended_ids
        .isin(curriculum_lesson_set)
    )


    # --------------------------------------------------------
    # 4. Find invalid records
    # --------------------------------------------------------

    invalid_lesson_rows = (
        invalid_missing_ids
        |
        ~lesson_exists
    )


    invalid_recommendations = (
        final_learning_path[
            invalid_lesson_rows
        ]
        [
            [
                "Student_ID",
                "Assessment_ID",
                "Recommended_Lesson_ID",
                "Recommended_Subject",
                "Recommended_Topic"
            ]
        ]
        .copy()
    )


    # --------------------------------------------------------
    # 5. Statistics
    # --------------------------------------------------------

    total_recommendations = len(
        final_learning_path
    )

    valid_recommendations = (
        ~invalid_lesson_rows
    ).sum()

    invalid_count = (
        invalid_lesson_rows.sum()
    )

    unique_curriculum_lessons = (
        len(curriculum_lesson_set)
    )

    unique_recommended_lessons = (
        recommended_ids[
            ~invalid_missing_ids
        ].nunique()
    )


    # --------------------------------------------------------
    # 6. Test result
    # --------------------------------------------------------

    passed = (
        invalid_count == 0
    )


    print(
        "Total recommendations:",
        total_recommendations
    )

    print(
        "Curriculum lessons:",
        unique_curriculum_lessons
    )

    print(
        "Unique recommended lessons:",
        unique_recommended_lessons
    )

    print(
        "Valid recommendations:",
        valid_recommendations
    )

    print(
        "Invalid recommendations:",
        invalid_count
    )


    if invalid_count > 0:

        print(
            "\nINVALID CURRICULUM "
            "LESSON RECOMMENDATIONS"
        )

        display(
            invalid_recommendations
        )

    else:

        print(
            "\n✓ All recommended lesson IDs "
            "exist in the curriculum dataset."
        )


    add_test(
        "T04",
        "Curriculum Lesson Validity",
        passed,
        (
            f"{valid_recommendations}/"
            f"{total_recommendations} recommendations "
            "have valid curriculum lesson IDs"
        )
    )


else:

    add_test(
        "T04",
        "Curriculum Lesson Validity",
        False,
        "Required datasets or columns unavailable"
    )

TEST 4 — CURRICULUM LESSON VALIDITY
Total recommendations: 30
Curriculum lessons: 20
Unique recommended lessons: 12
Valid recommendations: 30
Invalid recommendations: 0

✓ All recommended lesson IDs exist in the curriculum dataset.


In [133]:
# ============================================================
# STEP 9 — TEST LESSON SUBJECT CONSISTENCY
# ============================================================

print("=" * 70)
print("TEST 5 — LESSON SUBJECT CONSISTENCY")
print("=" * 70)


if (
    final_learning_path is not None
    and curriculum_clean is not None
):

    curriculum_lookup = (
        curriculum_clean[
            [
                "Lesson_ID",
                "Subject",
                "Topic",
                "Competency"
            ]
        ]
        .drop_duplicates(
            subset=["Lesson_ID"]
        )
        .copy()
    )


    curriculum_lookup["Lesson_ID"] = (
        curriculum_lookup["Lesson_ID"]
        .astype(str)
        .str.strip()
    )


    subject_check = (
        final_learning_path[
            [
                "Student_ID",
                "Assessment_ID",
                "Assessment_Subject",
                "Recommended_Lesson_ID",
                "Recommended_Subject",
                "Recommended_Topic",
                "Recommended_Competency"
            ]
        ]
        .copy()
    )


    subject_check[
        "Recommended_Lesson_ID"
    ] = (
        subject_check[
            "Recommended_Lesson_ID"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    subject_check = subject_check.merge(
        curriculum_lookup,
        left_on="Recommended_Lesson_ID",
        right_on="Lesson_ID",
        how="left",
        suffixes=(
            "_Recommendation",
            "_Curriculum"
        )
    )


    subject_check[
        "Recommendation_Subject_Normalized"
    ] = (
        subject_check[
            "Recommended_Subject"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )


    subject_check[
        "Curriculum_Subject_Normalized"
    ] = (
        subject_check[
            "Subject"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )


    subject_check[
        "Subject_Match"
    ] = (
        subject_check[
            "Recommendation_Subject_Normalized"
        ]
        ==
        subject_check[
            "Curriculum_Subject_Normalized"
        ]
    )


    subject_mismatches = subject_check[
        ~subject_check["Subject_Match"]
    ].copy()


    total_records = len(
        subject_check
    )

    mismatch_count = len(
        subject_mismatches
    )

    match_count = (
        subject_check["Subject_Match"]
        .sum()
    )


    passed = (
        mismatch_count == 0
    )


    print(
        "Total recommendations:",
        total_records
    )

    print(
        "Subject matches:",
        match_count
    )

    print(
        "Subject mismatches:",
        mismatch_count
    )


    if mismatch_count > 0:

        print("\nSUBJECT MISMATCHES")

        display(
            subject_mismatches[
                [
                    "Student_ID",
                    "Assessment_ID",
                    "Assessment_Subject",
                    "Recommended_Lesson_ID",
                    "Recommended_Subject",
                    "Subject"
                ]
            ]
        )

    else:

        print(
            "\n✓ All recommended lesson subjects "
            "match the curriculum."
        )


    add_test(
        "T05",
        "Lesson Subject Consistency",
        passed,
        (
            f"{match_count}/{total_records} "
            "recommendations have matching "
            "curriculum subjects"
        )
    )


else:

    add_test(
        "T05",
        "Lesson Subject Consistency",
        False,
        "Required datasets unavailable"
    )

TEST 5 — LESSON SUBJECT CONSISTENCY
Total recommendations: 30
Subject matches: 30
Subject mismatches: 0

✓ All recommended lesson subjects match the curriculum.


In [134]:
# ============================================================
# STEP 10 — DOMAIN-AWARE CURRICULUM VALIDATION
# ============================================================

print("=" * 70)
print("TEST 6 — DOMAIN-AWARE CURRICULUM VALIDATION")
print("=" * 70)


subject_domain_map = {

    "mathematics": [
        "mathematics"
    ],

    "english": [
        "english"
    ],

    "sinhala": [
        "sinhala"
    ],

    "environmental studies": [
        "environmental studies"
    ],

    "creativity": [
        "creative activities"
    ],

    "fine motor skills": [
        "creative activities",
        "life skills"
    ],

    "gross motor skills": [
        "life skills"
    ]
}


def get_allowed_subjects(
    assessment_subject
):

    subject = normalize_text(
        assessment_subject
    )

    return subject_domain_map.get(
        subject,
        [subject]
    )


if final_learning_path is not None:

    domain_check = []


    for _, row in (
        final_learning_path.iterrows()
    ):

        assessment_subject = (
            normalize_text(
                row[
                    "Assessment_Subject"
                ]
            )
        )


        recommended_subject = (
            normalize_text(
                row[
                    "Recommended_Subject"
                ]
            )
        )


        allowed_subjects = (
            get_allowed_subjects(
                assessment_subject
            )
        )


        valid = (
            recommended_subject
            in allowed_subjects
        )


        domain_check.append({

            "Student_ID":
                row["Student_ID"],

            "Assessment_ID":
                row["Assessment_ID"],

            "Assessment_Subject":
                row["Assessment_Subject"],

            "Recommended_Lesson_ID":
                row[
                    "Recommended_Lesson_ID"
                ],

            "Recommended_Subject":
                row[
                    "Recommended_Subject"
                ],

            "Valid_Domain":
                valid
        })


    domain_check = pd.DataFrame(
        domain_check
    )


    invalid_domain = (
        domain_check[
            ~domain_check["Valid_Domain"]
        ]
    )


    total_domain_records = len(
        domain_check
    )

    valid_domain_count = (
        domain_check[
            "Valid_Domain"
        ]
        .sum()
    )

    invalid_domain_count = len(
        invalid_domain
    )


    passed = (
        invalid_domain_count == 0
    )


    print(
        "Total recommendations:",
        total_domain_records
    )

    print(
        "Valid domain recommendations:",
        valid_domain_count
    )

    print(
        "Invalid domain recommendations:",
        invalid_domain_count
    )


    if invalid_domain_count > 0:

        print(
            "\nINVALID DOMAIN RECOMMENDATIONS"
        )

        display(
            invalid_domain
        )

    else:

        print(
            "\n✓ All recommendations "
            "belong to valid curriculum domains."
        )


    add_test(
        "T06",
        "Domain-Aware Validation",
        passed,
        (
            f"{valid_domain_count}/"
            f"{total_domain_records} recommendations "
            "have valid domain mappings"
        )
    )


else:

    add_test(
        "T06",
        "Domain-Aware Validation",
        False,
        "Final learning path unavailable"
    )

TEST 6 — DOMAIN-AWARE CURRICULUM VALIDATION
Total recommendations: 30
Valid domain recommendations: 30
Invalid domain recommendations: 0

✓ All recommendations belong to valid curriculum domains.


In [135]:
# ============================================================
# STEP 11 — ALIGNMENT SCORE VALIDATION
# ============================================================

print("=" * 70)
print("TEST 7 — ALIGNMENT SCORE VALIDATION")
print("=" * 70)


if (
    final_learning_path is not None
    and "Alignment_Score"
    in final_learning_path.columns
):

    alignment_scores = pd.to_numeric(
        final_learning_path[
            "Alignment_Score"
        ],
        errors="coerce"
    )


    missing_scores = (
        alignment_scores
        .isna()
        .sum()
    )


    infinite_scores = (
        np.isinf(
            alignment_scores
            .fillna(0)
        )
        .sum()
    )


    passed = (
        missing_scores == 0
        and
        infinite_scores == 0
    )


    print(
        "Total recommendations:",
        len(alignment_scores)
    )

    print(
        "Missing alignment scores:",
        missing_scores
    )

    print(
        "Infinite alignment scores:",
        infinite_scores
    )


    print(
        "\nAlignment score statistics:"
    )

    print(
        alignment_scores.describe()
    )


    add_test(
        "T07",
        "Alignment Score Validation",
        passed,
        (
            f"{missing_scores} missing, "
            f"{infinite_scores} infinite scores"
        )
    )


else:

    add_test(
        "T07",
        "Alignment Score Validation",
        False,
        "Alignment_Score column unavailable"
    )

TEST 7 — ALIGNMENT SCORE VALIDATION
Total recommendations: 30
Missing alignment scores: 0
Infinite alignment scores: 0

Alignment score statistics:
count     30.000000
mean     115.017333
std      143.719457
min      -40.000000
25%      -33.802500
50%      161.045000
75%      197.777500
max      352.500000
Name: Alignment_Score, dtype: float64


In [136]:
# ============================================================
# STEP 12 — RECOMMENDATION STATUS VALIDATION
# ============================================================

print("=" * 70)
print("TEST 8 — RECOMMENDATION STATUS VALIDATION")
print("=" * 70)


if (
    final_learning_path is not None
    and "Recommendation_Status"
    in final_learning_path.columns
    and "Alignment_Score"
    in final_learning_path.columns
):

    def expected_status(score):

        if pd.isna(score):

            return "Teacher Review Required"

        elif score >= 100:

            return "Recommended"

        else:

            return "Teacher Review Required"


    expected_status_values = (
        final_learning_path[
            "Alignment_Score"
        ]
        .apply(expected_status)
    )


    actual_status_values = (
        final_learning_path[
            "Recommendation_Status"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    status_match = (
        actual_status_values
        ==
        expected_status_values
    )


    status_errors = (
        final_learning_path[
            ~status_match
        ]
        .copy()
    )


    passed = (
        len(status_errors) == 0
    )


    print(
        "Correct recommendation statuses:",
        status_match.sum()
    )

    print(
        "Incorrect recommendation statuses:",
        len(status_errors)
    )


    if len(status_errors) > 0:

        display(
            status_errors[
                [
                    "Student_ID",
                    "Alignment_Score",
                    "Recommendation_Status"
                ]
            ]
        )


    add_test(
        "T08",
        "Recommendation Status Validation",
        passed,
        (
            f"{status_match.sum()}/"
            f"{len(status_match)} statuses correct"
        )
    )


else:

    add_test(
        "T08",
        "Recommendation Status Validation",
        False,
        "Required columns unavailable"
    )

TEST 8 — RECOMMENDATION STATUS VALIDATION
Correct recommendation statuses: 30
Incorrect recommendation statuses: 0


In [137]:
# ============================================================
# STEP 13 — ALIGNMENT LEVEL VALIDATION
# ============================================================

print("=" * 70)
print("TEST 9 — ALIGNMENT LEVEL VALIDATION")
print("=" * 70)


if (
    final_learning_path is not None
    and "Alignment_Level"
    in final_learning_path.columns
    and "Alignment_Score"
    in final_learning_path.columns
):

    def expected_alignment_level(score):

        if pd.isna(score):

            return "No Match"

        elif score >= 200:

            return "Excellent"

        elif score >= 100:

            return "Good"

        elif score >= 50:

            return "Moderate"

        else:

            return "Weak"


    expected_levels = (
        final_learning_path[
            "Alignment_Score"
        ]
        .apply(
            expected_alignment_level
        )
    )


    actual_levels = (
        final_learning_path[
            "Alignment_Level"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    level_match = (
        actual_levels
        ==
        expected_levels
    )


    level_errors = (
        final_learning_path[
            ~level_match
        ]
        .copy()
    )


    passed = (
        len(level_errors) == 0
    )


    print(
        "Correct alignment classifications:",
        level_match.sum()
    )

    print(
        "Incorrect classifications:",
        len(level_errors)
    )


    if len(level_errors) > 0:

        display(
            level_errors[
                [
                    "Student_ID",
                    "Alignment_Score",
                    "Alignment_Level"
                ]
            ]
        )


    add_test(
        "T09",
        "Alignment Level Validation",
        passed,
        (
            f"{level_match.sum()}/"
            f"{len(level_match)} classifications correct"
        )
    )


else:

    add_test(
        "T09",
        "Alignment Level Validation",
        False,
        "Required columns unavailable"
    )

TEST 9 — ALIGNMENT LEVEL VALIDATION
Correct alignment classifications: 30
Incorrect classifications: 0


In [138]:
# ============================================================
# STEP 14 — DUPLICATE RECOMMENDATION CHECK
# ============================================================

print("=" * 70)
print("TEST 10 — DUPLICATE RECOMMENDATION CHECK")
print("=" * 70)


if final_learning_path is not None:

    exact_duplicates = (
        final_learning_path
        .duplicated()
        .sum()
    )


    student_lesson_duplicates = (
        final_learning_path
        .duplicated(
            subset=[
                "Student_ID",
                "Recommended_Lesson_ID"
            ]
        )
        .sum()
    )


    passed = (
        exact_duplicates == 0
        and
        student_lesson_duplicates == 0
    )


    print(
        "Exact duplicate rows:",
        exact_duplicates
    )

    print(
        "Student + Lesson duplicate rows:",
        student_lesson_duplicates
    )


    if student_lesson_duplicates > 0:

        duplicate_rows = (
            final_learning_path[
                final_learning_path.duplicated(
                    subset=[
                        "Student_ID",
                        "Recommended_Lesson_ID"
                    ],
                    keep=False
                )
            ]
            .sort_values(
                [
                    "Student_ID",
                    "Recommended_Lesson_ID"
                ]
            )
        )


        print(
            "\nDUPLICATE STUDENT + LESSON "
            "RECOMMENDATIONS:"
        )

        display(
            duplicate_rows
        )


    add_test(
        "T10",
        "Duplicate Recommendation Check",
        passed,
        (
            f"{exact_duplicates} exact duplicates; "
            f"{student_lesson_duplicates} "
            "student-lesson duplicates"
        )
    )


else:

    add_test(
        "T10",
        "Duplicate Recommendation Check",
        False,
        "Final learning path unavailable"
    )

TEST 10 — DUPLICATE RECOMMENDATION CHECK
Exact duplicate rows: 0
Student + Lesson duplicate rows: 0


In [139]:
# ============================================================
# STEP 15 — LEARNING PATH ORDER VALIDATION
# ============================================================

print("=" * 70)
print("TEST 11 — LEARNING PATH ORDER VALIDATION")
print("=" * 70)


if (
    final_learning_path is not None
    and "Path_Order"
    in final_learning_path.columns
):

    path_order_check = (
        final_learning_path
        .groupby("Student_ID")[
            "Path_Order"
        ]
        .apply(
            lambda x:
            sorted(
                pd.to_numeric(
                    x,
                    errors="coerce"
                )
                .dropna()
                .astype(int)
                .tolist()
            )
            == [1, 2]
        )
    )


    invalid_students = (
        path_order_check[
            ~path_order_check
        ]
    )


    passed = (
        len(invalid_students) == 0
    )


    print(
        "Students checked:",
        len(path_order_check)
    )

    print(
        "Students with valid path order:",
        path_order_check.sum()
    )

    print(
        "Students with invalid path order:",
        len(invalid_students)
    )


    if len(invalid_students) > 0:

        print(
            "\nINVALID PATH ORDER:"
        )

        print(
            invalid_students
        )

    else:

        print(
            "\n✓ Every student has "
            "Path_Order [1, 2]."
        )


    add_test(
        "T11",
        "Learning Path Order",
        passed,
        (
            f"{len(path_order_check) - len(invalid_students)}/"
            f"{len(path_order_check)} students "
            "have valid path order"
        )
    )


else:

    add_test(
        "T11",
        "Learning Path Order",
        False,
        "Path_Order column unavailable"
    )

TEST 11 — LEARNING PATH ORDER VALIDATION
Students checked: 15
Students with valid path order: 15
Students with invalid path order: 0

✓ Every student has Path_Order [1, 2].


In [140]:
# ============================================================
# STEP 16 — PERFORMANCE LEVEL VALIDATION
# ============================================================

print("=" * 70)
print("TEST 12 — PERFORMANCE LEVEL VALIDATION")
print("=" * 70)


if (
    final_learning_path is not None
    and "Overall_Performance"
    in final_learning_path.columns
    and "Performance_Level"
    in final_learning_path.columns
):

    performance_scores = pd.to_numeric(
        final_learning_path[
            "Overall_Performance"
        ],
        errors="coerce"
    )


    def expected_performance_level(score):

        if pd.isna(score):

            return "Unknown"

        elif score >= 80:

            return "High"

        elif score >= 60:

            return "Moderate"

        else:

            return "Needs Support"


    expected_levels = (
        performance_scores
        .apply(
            expected_performance_level
        )
    )


    actual_levels = (
        final_learning_path[
            "Performance_Level"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    performance_match = (
        actual_levels
        ==
        expected_levels
    )


    performance_errors = (
        final_learning_path[
            ~performance_match
        ]
        .copy()
    )


    passed = (
        len(performance_errors) == 0
    )


    print(
        "Correct performance classifications:",
        performance_match.sum()
    )

    print(
        "Incorrect performance classifications:",
        len(performance_errors)
    )


    if len(performance_errors) > 0:

        display(
            performance_errors[
                [
                    "Student_ID",
                    "Overall_Performance",
                    "Performance_Level"
                ]
            ]
        )


    add_test(
        "T12",
        "Performance Level Validation",
        passed,
        (
            f"{performance_match.sum()}/"
            f"{len(performance_match)} "
            "performance classifications correct"
        )
    )


else:

    add_test(
        "T12",
        "Performance Level Validation",
        False,
        "Required columns unavailable"
    )

TEST 12 — PERFORMANCE LEVEL VALIDATION
Correct performance classifications: 30
Incorrect performance classifications: 0


In [141]:
# ============================================================
# STEP 17 — TEACHING STRATEGY VALIDATION
# ============================================================

print("=" * 70)
print("TEST 13 — TEACHING STRATEGY VALIDATION")
print("=" * 70)


if (
    final_learning_path is not None
    and "Performance_Level"
    in final_learning_path.columns
    and "Teaching_Strategy"
    in final_learning_path.columns
):

    strategy_results = []


    for _, row in (
        final_learning_path.iterrows()
    ):

        performance = normalize_text(
            row["Performance_Level"]
        )


        strategy = normalize_text(
            row["Teaching_Strategy"]
        )


        recommendation_status = (
            normalize_text(
                row.get(
                    "Recommendation_Status",
                    ""
                )
            )
        )


        # ----------------------------------------------------
        # Empty strategy is always invalid
        # ----------------------------------------------------

        if strategy == "":

            valid = False


        # ----------------------------------------------------
        # Teacher-review recommendations
        # ----------------------------------------------------

        elif (
            "teacher review"
            in strategy
        ):

            valid = True


        # ----------------------------------------------------
        # Needs Support
        # ----------------------------------------------------

        elif performance == "needs support":

            valid = (
                "guided"
                in strategy
                or
                "teacher support"
                in strategy
                or
                "individual support"
                in strategy
                or
                "support"
                in strategy
            )


        # ----------------------------------------------------
        # High performance
        # ----------------------------------------------------

        elif performance == "high":

            valid = (
                "independent"
                in strategy
                or
                "extension"
                in strategy
            )


        # ----------------------------------------------------
        # Moderate performance
        # ----------------------------------------------------

        elif performance == "moderate":

            valid = (
                "structured"
                in strategy
                or
                "reinforcement"
                in strategy
                or
                "guided"
                in strategy
                or
                "practice"
                in strategy
            )


        else:

            valid = False


        strategy_results.append(valid)


    final_learning_path[
        "Strategy_Valid"
    ] = strategy_results


    strategy_errors = (
        final_learning_path[
            ~final_learning_path[
                "Strategy_Valid"
            ]
        ]
        .copy()
    )


    valid_strategy_count = (
        final_learning_path[
            "Strategy_Valid"
        ]
        .sum()
    )


    total_strategy_records = len(
        final_learning_path
    )


    passed = (
        len(strategy_errors) == 0
    )


    print(
        "Total recommendations:",
        total_strategy_records
    )

    print(
        "Valid teaching strategies:",
        valid_strategy_count
    )

    print(
        "Potential strategy mismatches:",
        len(strategy_errors)
    )


    if len(strategy_errors) > 0:

        print(
            "\nTEACHING STRATEGY MISMATCHES"
        )

        display(
            strategy_errors[
                [
                    "Student_ID",
                    "Performance_Level",
                    "Recommendation_Status",
                    "Teaching_Strategy"
                ]
            ]
        )

    else:

        print(
            "\n✓ Teaching strategies are "
            "consistent with performance levels."
        )


    add_test(
        "T13",
        "Teaching Strategy Validation",
        passed,
        (
            f"{valid_strategy_count}/"
            f"{total_strategy_records} "
            "teaching strategies valid"
        )
    )


else:

    add_test(
        "T13",
        "Teaching Strategy Validation",
        False,
        "Required columns unavailable"
    )

TEST 13 — TEACHING STRATEGY VALIDATION
Total recommendations: 30
Valid teaching strategies: 30
Potential strategy mismatches: 0

✓ Teaching strategies are consistent with performance levels.


In [142]:
# ============================================================
# STEP 18 — STUDENT SUMMARY CONSISTENCY
# ============================================================

print("=" * 70)
print("TEST 14 — STUDENT SUMMARY CONSISTENCY")
print("=" * 70)


if (
    final_learning_path is not None
    and student_summary is not None
):

    # --------------------------------------------------------
    # Calculate summary directly from learning path
    # --------------------------------------------------------

    calculated_summary = (
        final_learning_path
        .groupby("Student_ID")
        .agg(

            Calculated_Number_of_Activities=(
                "Recommended_Lesson_ID",
                "count"
            ),

            Calculated_Average_Alignment_Score=(
                "Alignment_Score",
                "mean"
            ),

            Calculated_Overall_Performance=(
                "Overall_Performance",
                "first"
            ),

            Calculated_Performance_Level=(
                "Performance_Level",
                "first"
            )

        )
        .reset_index()
    )


    # --------------------------------------------------------
    # Merge with saved student summary
    # --------------------------------------------------------

    comparison = (
        student_summary.merge(
            calculated_summary,
            on="Student_ID",
            how="outer"
        )
    )


    # --------------------------------------------------------
    # Student ID check
    # --------------------------------------------------------

    student_id_match = (
        comparison[
            "Student_ID"
        ]
        .notna()
    )


    # --------------------------------------------------------
    # Activity count check
    # --------------------------------------------------------

    activity_count_match = (
        comparison[
            "Number_of_Learning_Activities"
        ]
        ==
        comparison[
            "Calculated_Number_of_Activities"
        ]
    )


    # --------------------------------------------------------
    # Alignment score check
    # --------------------------------------------------------

    saved_alignment = pd.to_numeric(
        comparison[
            "Average_Alignment_Score"
        ],
        errors="coerce"
    )


    calculated_alignment = pd.to_numeric(
        comparison[
            "Calculated_Average_Alignment_Score"
        ],
        errors="coerce"
    )


    alignment_match = np.isclose(
        saved_alignment,
        calculated_alignment,
        atol=0.01,
        equal_nan=False
    )


    # --------------------------------------------------------
    # Performance level check
    # --------------------------------------------------------

    performance_match = (
        comparison[
            "Performance_Level"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        ==
        comparison[
            "Calculated_Performance_Level"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # Overall performance check
    # --------------------------------------------------------

    saved_overall = pd.to_numeric(
        comparison[
            "Overall_Performance"
        ],
        errors="coerce"
    )


    calculated_overall = pd.to_numeric(
        comparison[
            "Calculated_Overall_Performance"
        ],
        errors="coerce"
    )


    overall_performance_match = np.isclose(
        saved_overall,
        calculated_overall,
        atol=0.01,
        equal_nan=False
    )


    # --------------------------------------------------------
    # Final consistency
    # --------------------------------------------------------

    comparison[
        "Summary_Match"
    ] = (
        student_id_match
        &
        activity_count_match
        &
        alignment_match
        &
        performance_match
        &
        overall_performance_match
    )


    summary_errors = comparison[
        ~comparison["Summary_Match"]
    ]


    passed = (
        len(summary_errors) == 0
    )


    print(
        "Students checked:",
        len(comparison)
    )

    print(
        "Summary matches:",
        comparison[
            "Summary_Match"
        ].sum()
    )

    print(
        "Summary mismatches:",
        len(summary_errors)
    )


    if len(summary_errors) > 0:

        print(
            "\nSTUDENT SUMMARY MISMATCHES"
        )

        display(
            summary_errors
        )

    else:

        print(
            "\n✓ Student summary is "
            "consistent with the learning path."
        )


    add_test(
        "T14",
        "Student Summary Consistency",
        passed,
        (
            f"{len(comparison) - len(summary_errors)}/"
            f"{len(comparison)} students "
            "have consistent summaries"
        )
    )


else:

    add_test(
        "T14",
        "Student Summary Consistency",
        False,
        "Required datasets unavailable"
    )

TEST 14 — STUDENT SUMMARY CONSISTENCY
Students checked: 15
Summary matches: 15
Summary mismatches: 0

✓ Student summary is consistent with the learning path.


In [143]:
# ============================================================
# STEP 19 — ASSESSMENT ID VALIDITY
# ============================================================

print("=" * 70)
print("TEST 15 — ASSESSMENT ID VALIDITY")
print("=" * 70)


if (
    final_learning_path is not None
    and assessment_clean is not None
):

    assessment_ids = set(
        assessment_clean[
            "Assessment_ID"
        ]
        .dropna()
        .astype(str)
        .str.strip()
    )


    recommended_assessment_ids = (
        final_learning_path[
            "Assessment_ID"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    missing_assessment_rows = (
        recommended_assessment_ids
        .eq("")
    )


    invalid_assessment_rows = (
        ~recommended_assessment_ids
        .isin(assessment_ids)
    )


    invalid_rows = (
        missing_assessment_rows
        |
        invalid_assessment_rows
    )


    invalid_assessments = (
        final_learning_path[
            invalid_rows
        ]
        [
            [
                "Student_ID",
                "Assessment_ID",
                "Assessment_Subject",
                "Assessment_Type"
            ]
        ]
        .copy()
    )


    total_records = len(
        final_learning_path
    )


    valid_records = (
        ~invalid_rows
    ).sum()


    invalid_count = (
        invalid_rows.sum()
    )


    passed = (
        invalid_count == 0
    )


    print(
        "Assessment IDs in source dataset:",
        len(assessment_ids)
    )

    print(
        "Assessment records in learning path:",
        total_records
    )

    print(
        "Valid assessment references:",
        valid_records
    )

    print(
        "Invalid assessment references:",
        invalid_count
    )


    if invalid_count > 0:

        print(
            "\nINVALID ASSESSMENT REFERENCES"
        )

        display(
            invalid_assessments
        )

    else:

        print(
            "\n✓ All Assessment_ID values "
            "exist in assessment_clean.csv."
        )


    add_test(
        "T15",
        "Assessment ID Validity",
        passed,
        (
            f"{valid_records}/"
            f"{total_records} assessment "
            "references are valid"
        )
    )


else:

    add_test(
        "T15",
        "Assessment ID Validity",
        False,
        "Required datasets unavailable"
    )

TEST 15 — ASSESSMENT ID VALIDITY
Assessment IDs in source dataset: 30
Assessment records in learning path: 30
Valid assessment references: 30
Invalid assessment references: 0

✓ All Assessment_ID values exist in assessment_clean.csv.


In [144]:
# ============================================================
# STEP 20 — FINAL STUDENT-WISE VALIDATION
# ============================================================

print("=" * 70)
print("FINAL STUDENT-WISE VALIDATION")
print("=" * 70)


if final_learning_path is not None:

    student_validation = (
        final_learning_path
        .groupby("Student_ID")
        .agg(

            Number_of_Recommendations=(
                "Recommended_Lesson_ID",
                "count"
            ),

            Average_Alignment_Score=(
                "Alignment_Score",
                "mean"
            ),

            Minimum_Alignment_Score=(
                "Alignment_Score",
                "min"
            ),

            Maximum_Alignment_Score=(
                "Alignment_Score",
                "max"
            ),

            Unique_Lessons=(
                "Recommended_Lesson_ID",
                "nunique"
            )

        )
        .reset_index()
    )


    student_validation[
        "Expected_Recommendations"
    ] = 2


    student_validation[
        "Count_Check"
    ] = (
        student_validation[
            "Number_of_Recommendations"
        ]
        ==
        student_validation[
            "Expected_Recommendations"
        ]
    )


    student_validation[
        "Unique_Lesson_Check"
    ] = (
        student_validation[
            "Unique_Lessons"
        ]
        ==
        student_validation[
            "Number_of_Recommendations"
        ]
    )


    display(
        student_validation
    )


else:

    print(
        "Final learning path unavailable."
    )

FINAL STUDENT-WISE VALIDATION


,Student_ID,Number_of_Recommendations,Average_Alignment_Score,Minimum_Alignment_Score,Maximum_Alignment_Score,Unique_Lessons,Expected_Recommendations,Count_Check,Unique_Lesson_Check
0,ST001,2,263.485,174.47,352.50,2,2,True,True
1,ST002,2,63.160,-40.00,166.32,2,2,True,True
2,ST003,2,61.880,-39.57,163.33,2,2,True,True
3,ST004,2,79.960,-37.37,197.29,2,2,True,True
4,ST005,2,124.645,-10.71,260.00,2,2,True,True
5,ST006,2,243.875,169.29,318.46,2,2,True,True
6,ST007,2,-33.450,-36.36,-30.54,2,2,True,True
7,ST008,2,192.535,171.21,213.86,2,2,True,True
8,ST009,2,161.540,-26.92,350.00,2,2,True,True
9,ST010,2,64.030,-34.35,162.41,2,2,True,True


In [145]:
# ============================================================
# STEP 21 — DISPLAY FINAL PERSONALIZED LEARNING PATHS
# ============================================================

print("=" * 70)
print("FINAL PERSONALIZED LEARNING PATHS")
print("=" * 70)


display_columns = [

    "Student_ID",

    "Path_Order",

    "Assessment_Subject",

    "Assessment_Type",

    "Recommended_Lesson_ID",

    "Recommended_Subject",

    "Recommended_Topic",

    "Recommended_Competency",

    "Learning_Objective",

    "Difficulty_Level",

    "Classroom_Activities",

    "Alignment_Score",

    "Alignment_Level",

    "Performance_Level",

    "Teaching_Strategy",

    "Recommendation_Status",

    "Recommended_Duration",

    "Teacher_Action"
]


available_display_columns = [
    col
    for col in display_columns
    if col in final_learning_path.columns
]


display(
    final_learning_path[
        available_display_columns
    ]
    .sort_values(
        [
            "Student_ID",
            "Path_Order"
        ]
    )
    .reset_index(drop=True)
)

FINAL PERSONALIZED LEARNING PATHS


,Student_ID,Path_Order,Assessment_Subject,Assessment_Type,Recommended_Lesson_ID,Recommended_Subject,Recommended_Topic,Recommended_Competency,Learning_Objective,Difficulty_Level,Classroom_Activities,Alignment_Score,Alignment_Level,Performance_Level,Teaching_Strategy,Recommendation_Status,Recommended_Duration,Teacher_Action
0,ST001,1,Mathematics,Number Recognition,LES021,Mathematics,Number Recognition,Number Sense,Identify numbers 11-50 correctly,Basic,Number chart activities,352.50,Excellent,Moderate,Structured Practice and Reinforcement,Recommended,20 minutes,Provide structured practice and monitor progress.
1,ST001,2,English,Phonics,LES022,English,Simple Words,Reading Skills,Read and identify simple CVC words,Basic,Word card exercises,174.47,Good,Moderate,Structured Practice and Reinforcement,Recommended,20 minutes,Provide structured practice and monitor progress.
2,ST002,1,Mathematics,Counting Objects,LES030,Mathematics,Addition Within 20,Calculation Skills,Solve simple addition problems up to 20,Basic,Addition using objects,166.32,Good,Moderate,Structured Practice and Reinforcement,Recommended,20 minutes,Provide structured practice and monitor progress.
3,ST002,2,Fine Motor Skills,Pencil Control,LES040,Creative Activities,Craft Work,Creativity Skills,Create simple crafts using materials,Basic,Paper craft activities,-40.00,Weak,Moderate,Teacher Review and Individual Support,Teacher Review Required,20 minutes,Teacher should review the student's skill gap ...
4,ST003,1,English,Letter Recognition,LES026,English,Sentence Formation,Writing Skills,Create simple sentences using words,Intermediate,Sentence building cards,163.33,Good,Moderate,Structured Practice and Reinforcement,Recommended,25 minutes,Provide structured practice and monitor progress.
5,ST003,2,Creativity,Drawing Shapes,LES034,Creative Activities,Colouring,Creativity Skills,Use colours appropriately in drawings,Beginner,Colouring worksheets,-39.57,Weak,Moderate,Teacher Review and Individual Support,Teacher Review Required,15 minutes,Teacher should review the student's skill gap ...
6,ST004,1,Mathematics,Shape Recognition,LES021,Mathematics,Number Recognition,Number Sense,Identify numbers 11-50 correctly,Basic,Number chart activities,197.29,Good,Moderate,Structured Practice and Reinforcement,Recommended,20 minutes,Provide structured practice and monitor progress.
7,ST004,2,Gross Motor Skills,Balance and Movement,LES039,Life Skills,Sharing and Caring,Social Skills,Understand sharing and helping others,Beginner,Group activities,-37.37,Weak,Moderate,Teacher Review and Individual Support,Teacher Review Required,15 minutes,Teacher should review the student's skill gap ...
8,ST005,1,Environmental Studies,Plants and Animals,LES029,Environmental Studies,Plants,Environmental Awareness,Identify common plants around home and school,Beginner,Plant identification activity,260.00,Excellent,High,Independent Practice with Extension,Recommended,15 minutes,Allow independent practice and provide extensi...
9,ST005,2,Creativity,Paper Craft,LES040,Creative Activities,Craft Work,Creativity Skills,Create simple crafts using materials,Basic,Paper craft activities,-10.71,Weak,High,Teacher Review and Individual Support,Teacher Review Required,20 minutes,Teacher should review the student's skill gap ...


In [146]:
# ============================================================
# STEP 22 — OVERALL TEST RESULTS
# ============================================================

print("=" * 70)
print("OVERALL RECOMMENDATION TEST RESULTS")
print("=" * 70)


test_results_df = pd.DataFrame(
    test_results
)


display(
    test_results_df
)


total_tests = len(
    test_results_df
)


passed_tests = (
    test_results_df[
        "Status"
    ]
    == "PASS"
).sum()


failed_tests = (
    test_results_df[
        "Status"
    ]
    == "FAIL"
).sum()


print("\n" + "=" * 70)
print("TEST SUMMARY")
print("=" * 70)


print(
    "Total tests :",
    total_tests
)

print(
    "Passed      :",
    passed_tests
)

print(
    "Failed      :",
    failed_tests
)


if failed_tests == 0:

    print(
        "\n✓ ALL RECOMMENDATION TESTS PASSED"
    )

else:

    print(
        "\n⚠ SOME TESTS FAILED — "
        "REVIEW THE RESULTS ABOVE"
    )

OVERALL RECOMMENDATION TEST RESULTS


,Test_ID,Test_Name,Status,Details
0,T01,Student Coverage,PASS,15 students covered by the learning path
1,T02,Learning Path Count,PASS,30 records; 0 students with incorrect counts
2,T03,Required Dataset Columns,PASS,All required columns available
3,T04,Curriculum Lesson Validity,PASS,30/30 recommendations have valid curriculum le...
4,T05,Lesson Subject Consistency,PASS,30/30 recommendations have matching curriculum...
5,T06,Domain-Aware Validation,PASS,30/30 recommendations have valid domain mappings
6,T07,Alignment Score Validation,PASS,"0 missing, 0 infinite scores"
7,T08,Recommendation Status Validation,PASS,30/30 statuses correct
8,T09,Alignment Level Validation,PASS,30/30 classifications correct
9,T10,Duplicate Recommendation Check,PASS,0 exact duplicates; 0 student-lesson duplicates



TEST SUMMARY
Total tests : 15
Passed      : 15
Failed      : 0

✓ ALL RECOMMENDATION TESTS PASSED


In [147]:
# ============================================================
# STEP 23 — RECOMMENDATION QUALITY STATISTICS
# ============================================================

print("=" * 70)
print("RECOMMENDATION QUALITY STATISTICS")
print("=" * 70)


if final_learning_path is not None:

    print(
        "\nTotal learning-path records:"
    )

    print(
        len(final_learning_path)
    )


    print(
        "\nNumber of students:"
    )

    print(
        final_learning_path[
            "Student_ID"
        ]
        .nunique()
    )


    print(
        "\nUnique recommended lessons:"
    )

    print(
        final_learning_path[
            "Recommended_Lesson_ID"
        ]
        .nunique()
    )


    print(
        "\nRecommendation Status:"
    )

    print(
        final_learning_path[
            "Recommendation_Status"
        ]
        .value_counts()
    )


    print(
        "\nAlignment Level:"
    )

    print(
        final_learning_path[
            "Alignment_Level"
        ]
        .value_counts()
    )


    print(
        "\nPerformance Level:"
    )

    print(
        final_learning_path[
            "Performance_Level"
        ]
        .value_counts()
    )


    print(
        "\nTeaching Strategy:"
    )

    print(
        final_learning_path[
            "Teaching_Strategy"
        ]
        .value_counts()
    )


    print(
        "\nAverage Alignment Score:"
    )

    print(
        round(
            pd.to_numeric(
                final_learning_path[
                    "Alignment_Score"
                ],
                errors="coerce"
            ).mean(),
            2
        )
    )


    print(
        "\nRecommendation Coverage:"
    )

    print(
        f"{len(final_learning_path)} "
        "learning-path records generated."
    )

RECOMMENDATION QUALITY STATISTICS

Total learning-path records:
30

Number of students:
15

Unique recommended lessons:
12

Recommendation Status:
Recommendation_Status
Recommended                17
Teacher Review Required    13
Name: count, dtype: int64

Alignment Level:
Alignment_Level
Weak         13
Good         10
Excellent     7
Name: count, dtype: int64

Performance Level:
Performance_Level
Moderate         20
High              6
Needs Support     4
Name: count, dtype: int64

Teaching Strategy:
Teaching_Strategy
Teacher Review and Individual Support    13
Structured Practice and Reinforcement    10
Independent Practice with Extension       4
Guided Practice with Teacher Support      3
Name: count, dtype: int64

Average Alignment Score:
115.02

Recommendation Coverage:
30 learning-path records generated.


In [148]:
# ============================================================
# STEP 24 — SAVE VALIDATED OUTPUTS
# ============================================================

print("=" * 70)
print("SAVING VALIDATED RECOMMENDATION OUTPUTS")
print("=" * 70)


output_path = DATA_PATH


# ------------------------------------------------------------
# 1. Save test results
# ------------------------------------------------------------

test_results_file = (
    output_path
    / "recommendation_test_results.csv"
)


test_results_df.to_csv(
    test_results_file,
    index=False
)


print(
    "✓ Saved:",
    test_results_file
)


# ------------------------------------------------------------
# 2. Save student-wise validation
# ------------------------------------------------------------

student_validation_file = (
    output_path
    / "student_learning_path_validation.csv"
)


student_validation.to_csv(
    student_validation_file,
    index=False
)


print(
    "✓ Saved:",
    student_validation_file
)


# ------------------------------------------------------------
# 3. Define clean production columns
# ------------------------------------------------------------

final_columns = [

    "Student_ID",

    "Path_Order",

    "Learning_Stage",

    "Assessment_ID",

    "Assessment_Subject",

    "Assessment_Type",

    "Recommended_Lesson_ID",

    "Recommended_Subject",

    "Recommended_Topic",

    "Recommended_Competency",

    "Learning_Objective",

    "Difficulty_Level",

    "Required_Skills",

    "Classroom_Activities",

    "Alignment_Score",

    "Alignment_Level",

    "Recommendation_Status",

    "Performance_Level",

    "Overall_Performance",

    "Teaching_Strategy",

    "Recommended_Duration",

    "Teacher_Action"
]


available_final_columns = [
    col
    for col in final_columns
    if col in final_learning_path.columns
]


validated_learning_path = (
    final_learning_path[
        available_final_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. Save clean validated learning path
# ------------------------------------------------------------

validated_path_file = (
    output_path
    / "validated_personalized_learning_paths.csv"
)


validated_learning_path.to_csv(
    validated_path_file,
    index=False
)


print(
    "✓ Saved:",
    validated_path_file
)


print(
    "\nValidated output shape:",
    validated_learning_path.shape
)

SAVING VALIDATED RECOMMENDATION OUTPUTS
✓ Saved: ../data/processed/recommendation_test_results.csv
✓ Saved: ../data/processed/student_learning_path_validation.csv
✓ Saved: ../data/processed/validated_personalized_learning_paths.csv

Validated output shape: (30, 22)


In [149]:
# ============================================================
# STEP 25 — FINAL COMPLETION
# ============================================================

print("\n" + "=" * 70)
print("SMARTPLAY RECOMMENDATION TESTING COMPLETED")
print("=" * 70)


print("\nFinal Dataset Statistics")
print("-" * 70)


print(
    "Students:",
    validated_learning_path[
        "Student_ID"
    ].nunique()
)


print(
    "Learning-path records:",
    len(validated_learning_path)
)


print(
    "Unique recommended lessons:",
    validated_learning_path[
        "Recommended_Lesson_ID"
    ].nunique()
)


print(
    "\nTest Results"
)

print(
    "Passed:",
    passed_tests
)

print(
    "Failed:",
    failed_tests
)

print(
    "Total:",
    total_tests
)


if failed_tests == 0:

    print(
        "\n✓ SYSTEM VALIDATION STATUS: PASSED"
    )

    print(
        "✓ All recommendation tests passed."
    )

else:

    print(
        "\n⚠ SYSTEM VALIDATION STATUS: "
        "REVIEW REQUIRED"
    )


print("\nOutput Files")
print("-" * 70)

print(
    "1. recommendation_test_results.csv"
)

print(
    "2. student_learning_path_validation.csv"
)

print(
    "3. validated_personalized_learning_paths.csv"
)


print("\n" + "=" * 70)
print("END OF RECOMMENDATION TESTING")
print("=" * 70)


SMARTPLAY RECOMMENDATION TESTING COMPLETED

Final Dataset Statistics
----------------------------------------------------------------------
Students: 15
Learning-path records: 30
Unique recommended lessons: 12

Test Results
Passed: 15
Failed: 0
Total: 15

✓ SYSTEM VALIDATION STATUS: PASSED
✓ All recommendation tests passed.

Output Files
----------------------------------------------------------------------
1. recommendation_test_results.csv
2. student_learning_path_validation.csv
3. validated_personalized_learning_paths.csv

END OF RECOMMENDATION TESTING
